<a href="https://colab.research.google.com/github/Sabari19-adda/An-Efficient-SFSF-Knowledge-Distillation-Framework/blob/main/data_aug_KD_incepRes.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q kaggle
from google.colab import files
files.upload()  # Upload kaggle.json when prompted

# Kaggle API setup
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

# Download dataset
!kaggle datasets download -d uraninjo/augmented-alzheimer-mri-dataset-v2
!unzip -q augmented-alzheimer-mri-dataset-v2.zip -d data

from google.colab import drive
drive.mount('/content/drive')

# ===================== IMPORTS =====================
import os, shutil, numpy as np, pandas as pd
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Model, load_model
from tensorflow.keras.layers import (Input, SeparableConv2D, BatchNormalization,
                                     MaxPooling2D, GlobalAveragePooling2D,
                                     Dense, Dropout, Softmax)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.losses import CategoricalCrossentropy, KLDivergence
from tensorflow.keras.utils import Sequence
from sklearn.metrics import precision_score, recall_score, f1_score, confusion_matrix

# ===================== PATHS & BASIC SETTINGS =====================
base_dir = 'data/data'   # dataset root created by unzip above
combined_dir = 'combined_data'
os.makedirs(combined_dir + '/all', exist_ok=True)

# Merge train + val into combined_data/all/<class>
classes = os.listdir(os.path.join(base_dir, 'train'))
for cls in classes:
    os.makedirs(f"{combined_dir}/all/{cls}", exist_ok=True)
    for split in ['train', 'val']:
        src = os.path.join(base_dir, split, cls)
        if not os.path.exists(src):
            continue
        for img in os.listdir(src):
            shutil.copy(os.path.join(src, img), f"{combined_dir}/all/{cls}")

img_size = (224, 224)
batch_size = 32
seed = 42
np.random.seed(seed)
tf.random.set_seed(seed)

# ===================== ImageDataGenerators =====================
# datagen for training/val with augmentation + validation_split
datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.2,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.2
)

train_gen = datagen.flow_from_directory(
    combined_dir + '/all',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='training',
    shuffle=True,
    seed=seed
)

val_gen = datagen.flow_from_directory(
    combined_dir + '/all',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    subset='validation',
    shuffle=False,
    seed=seed
)

# filelist_gen used to gather filepaths & teacher predictions (shuffle=False)
filelist_datagen = ImageDataGenerator(rescale=1./255)
filelist_gen = filelist_datagen.flow_from_directory(
    combined_dir + '/all',
    target_size=img_size,
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

# collect filepaths and labels (one-hot)
filepaths = filelist_gen.filepaths
labels_onehot = tf.keras.utils.to_categorical(filelist_gen.labels, num_classes=len(filelist_gen.class_indices))
print(f"Total files discovered: {len(filepaths)}")

# ===================== STUDENT MODEL (same as your architecture) =====================
def build_separable_model(input_shape=(224, 224, 3), num_classes=4):
    inputs = Input(shape=input_shape)
    x = SeparableConv2D(32, 3, padding='same', activation='relu')(inputs)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)
    x = SeparableConv2D(64, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)
    x = SeparableConv2D(128, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)
    x = SeparableConv2D(256, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)
    x = SeparableConv2D(256, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)
    x = SeparableConv2D(512, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)
    x = SeparableConv2D(512, 3, padding='same', activation='relu')(x)
    x = BatchNormalization()(x); x = MaxPooling2D()(x)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.5)(x)
    outputs = Dense(len(filelist_gen.class_indices), activation='softmax')(x)
    model = Model(inputs, outputs, name="Student_SeparableConv")
    model.compile(optimizer=Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

student = build_separable_model()
print("✅ Student model built")
student.summary()

Saving kaggle.json to kaggle.json
Dataset URL: https://www.kaggle.com/datasets/uraninjo/augmented-alzheimer-mri-dataset-v2
License(s): GNU Lesser General Public License 3.0
 81% 308M/379M [00:00<00:00, 847MB/s] 
100% 379M/379M [00:00<00:00, 628MB/s]
Mounted at /content/drive
Found 32308 images belonging to 4 classes.
Found 8076 images belonging to 4 classes.
Found 40384 images belonging to 4 classes.
Total files discovered: 40384
✅ Student model built


Model: "Student_SeparableConv"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_layer (InputLayer)        │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d                │ (None, 224, 224, 32)   │           155 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_1              │ (None, 112, 112, 64)   │         2,400 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_2              │ (None, 56, 56, 128)    │         8,896 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_3              │ (None, 28, 28, 256)    │        34,176 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 28, 28, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 14, 14, 256)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_4              │ (None, 14, 14, 256)    │        68,096 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 14, 14, 256)    │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 7, 7, 256)      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ separable_conv2d_5              │ (None, 7, 7, 512)      │       133,888 │
│ (SeparableConv2D)               │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 7, 7, 512)      │         2,048 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 3, 3, 512)      │             

 Total params: 655,295 (2.50 MB)

 Trainable params: 651,263 (2.48 MB)

 Non-trainable params: 4,032 (15.75 KB)

In [ ]:
# ---------- Put this cell at the top, before load_model calls ----------

import os
import json
import h5py
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.utils import get_custom_objects
from tensorflow.keras.layers import Layer

# Robust CustomScaleLayer to match saved model (your provided implementation)
class CustomScaleLayer(tf.keras.layers.Layer):
    def __init__(self, scale=1.0, **kwargs):
        super(CustomScaleLayer, self).__init__(**kwargs)
        self.scale = float(scale)

    def build(self, input_shape):
        super(CustomScaleLayer, self).build(input_shape)

    def call(self, inputs, **kwargs):
        # If inputs is a list/tuple, treat first as main branch, scale and add extras
        if isinstance(inputs, (list, tuple)):
            main = tf.convert_to_tensor(inputs[0])
            out = tf.multiply(main, tf.cast(self.scale, main.dtype))
            for extra in inputs[1:]:
                out = tf.add(out, tf.convert_to_tensor(extra))
            return out
        else:
            t = tf.convert_to_tensor(inputs)
            return tf.multiply(t, tf.cast(self.scale, t.dtype))

    def get_config(self):
        cfg = super(CustomScaleLayer, self).get_config()
        cfg.update({"scale": self.scale})
        return cfg

# Register the custom layer globally so load_model can find it
get_custom_objects().update({'CustomScaleLayer': CustomScaleLayer})

# Optional: helper to show layer class names stored in an H5 model file.
# This helps detect other custom layers/classes you must register.
def inspect_h5_layer_classes(h5_path):
    try:
        with h5py.File(h5_path, 'r') as f:
            # Keras may store model_config either in attrs or as dataset; handle both
            if 'model_config' in f.attrs:
                config = json.loads(f.attrs['model_config'].decode('utf-8'))
            elif 'model_config' in f:
                raw = f['model_config'][()]
                if isinstance(raw, bytes):
                    raw = raw.decode('utf-8')
                config = json.loads(raw)
            else:
                print("No model_config found in H5 file; it may be a SavedModel or different format.")
                return

        # Walk config JSON to collect all class_name occurrences
        class_names = set()
        def walk(obj):
            if isinstance(obj, dict):
                if 'class_name' in obj:
                    class_names.add(obj['class_name'])
                for v in obj.values():
                    walk(v)
            elif isinstance(obj, list):
                for item in obj:
                    walk(item)

        walk(config)
        print("Layer / class names found in model_config (unique):")
        for name in sorted(class_names):
            print(" -", name)
    except Exception as e:
        print("Could not inspect H5 file:", e)

# Example usage of the inspector (uncomment to run):
# inspect_h5_layer_classes('/content/drive/MyDrive/Alzheimer_Models/InceptionResNetV2_best_model.h5')

# Now load your teacher model (use compile=False to avoid needing custom losses/metrics at load time)
teacher_path = "/content/drive/MyDrive/Alzheimer_Models/InceptionResNetV2_best_model.h5"  # adjust if needed
try:
    teacher = load_model(teacher_path, compile=False)  # compile=False avoids deserializing custom losses/metrics
    teacher.trainable = False
    print("✅ Teacher loaded and frozen (compile=False).")
except ValueError as e:
    # If error persists, print it and also inspect model file for custom classes
    print("❌ Error loading model:", e)
    print("\nAttempting to list class names stored in the model file to detect missing custom objects...")
    inspect_h5_layer_classes(teacher_path)
    raise

# If you later need the model compiled with optimizer/metrics, compile after loading:
# teacher.compile(optimizer=tf.keras.optimizers.Adam(1e-4), loss='categorical_crossentropy', metrics=['accuracy'])


✅ Teacher loaded and frozen (compile=False).


In [ ]:
# ===================== LOAD TEACHER =====================
teacher_path = "/content/drive/MyDrive/Alzheimer_Models/InceptionResNetV2_best_model.h5"  # adjust if different
teacher = load_model(teacher_path)
teacher.trainable = False
print("✅ Teacher loaded and frozen")

# ===================== GENERATE TEACHER SOFT LABELS (for all files, aligned to filepaths order) =====================
print("🔹 Generating teacher predictions for all files (this may take time)...")
teacher_train_probs = teacher.predict(filelist_gen, verbose=1)  # shape (N, num_classes)
print("✅ Teacher predictions generated")

# save teacher probs and mapping for reproducibility and future run
save_dir = "/content/drive/MyDrive/alz_kd_results_run1"
os.makedirs(save_dir, exist_ok=True)

pd.DataFrame({
    'filepath': filepaths,
    'class_index': filelist_gen.labels
}).to_csv(os.path.join(save_dir, 'filelist_mapping.csv'), index=False)

np.save(os.path.join(save_dir, 'teacher_probs_all.npy'), teacher_train_probs)
print(f"Saved teacher predictions and mapping to: {save_dir}")

# ===================== SPLIT INDICES (80/20) — reproducible using seed =====================
num_samples = len(filepaths)
indices = np.arange(num_samples)
np.random.seed(seed)
np.random.shuffle(indices)

split = int(num_samples * 0.8)
train_idx = indices[:split]
val_idx = indices[split:]

train_filepaths = [filepaths[i] for i in train_idx]
train_labels = labels_onehot[train_idx]
train_teacher_probs = teacher_train_probs[train_idx]

val_filepaths = [filepaths[i] for i in val_idx]
val_labels = labels_onehot[val_idx]
val_teacher_probs = teacher_train_probs[val_idx]

print(f"Train samples: {len(train_filepaths)}, Val samples: {len(val_filepaths)}")

# ===================== KDSequence (yields ([images, teacher_probs], labels)) =====================
class KDSequence(Sequence):
    def __init__(self, filepaths, labels, teacher_probs, batch_size=32, img_size=(224,224), shuffle=True):
        self.filepaths = np.array(filepaths)
        self.labels = np.array(labels)
        self.teacher_probs = np.array(teacher_probs)
        self.batch_size = batch_size
        self.img_size = img_size
        self.indices = np.arange(len(self.filepaths))
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.filepaths) / float(self.batch_size)))

    def __getitem__(self, idx):
        batch_indices = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]
        batch_files = self.filepaths[batch_indices]
        batch_labels = self.labels[batch_indices]
        batch_teacher = self.teacher_probs[batch_indices]

        batch_images = np.zeros((len(batch_files), self.img_size[0], self.img_size[1], 3), dtype=np.float32)
        for i, p in enumerate(batch_files):
            img = load_img(p, target_size=self.img_size)
            arr = img_to_array(img) / 255.0
            batch_images[i] = arr

        return (batch_images.astype(np.float32), batch_teacher.astype(np.float32)), batch_labels.astype(np.float32)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

train_seq = KDSequence(train_filepaths, train_labels, train_teacher_probs, batch_size=batch_size, img_size=img_size, shuffle=True)
val_seq = KDSequence(val_filepaths, val_labels, val_teacher_probs, batch_size=batch_size, img_size=img_size, shuffle=False)

# ===================== DISTILLER (custom Model) =====================
class Distiller(tf.keras.Model):
    def __init__(self, student, teacher, alpha=0.5, temperature=3.0):
        super(Distiller, self).__init__()
        self.student = student
        self.teacher = teacher
        self.alpha = alpha
        self.temperature = temperature
        self.student_loss_tracker = tf.keras.metrics.Mean(name='student_loss')
        self.kd_loss_tracker = tf.keras.metrics.Mean(name='kd_loss')
        self.total_loss_tracker = tf.keras.metrics.Mean(name='loss')
        self.acc_metric = tf.keras.metrics.CategoricalAccuracy(name='accuracy')

    @property
    def metrics(self):
        return [self.total_loss_tracker, self.student_loss_tracker, self.kd_loss_tracker, self.acc_metric]

    def compile(self, optimizer, student_loss_fn=None, kd_loss_fn=None, **kwargs):
        super(Distiller, self).compile(**kwargs)
        self.optimizer = optimizer
        self.student_loss_fn = student_loss_fn or CategoricalCrossentropy(from_logits=True)
        self.kd_loss_fn = kd_loss_fn or KLDivergence()

    def train_step(self, data):
        (x, teacher_probs), y = data
        with tf.GradientTape() as tape:
            student_logits = self.student(x, training=True)
            s_loss = self.student_loss_fn(y, student_logits)
            T = tf.cast(self.temperature, tf.float32)
            teacher_soft = tf.nn.softmax(teacher_probs / T, axis=1)
            student_soft = tf.nn.softmax(student_logits / T, axis=1)
            kd_loss = self.kd_loss_fn(teacher_soft, student_soft) * (T * T)
            loss = self.alpha * s_loss + (1.0 - self.alpha) * kd_loss
        grads = tape.gradient(loss, self.student.trainable_variables)
        self.optimizer.apply_gradients(zip(grads, self.student.trainable_variables))
        self.total_loss_tracker.update_state(loss)
        self.student_loss_tracker.update_state(s_loss)
        self.kd_loss_tracker.update_state(kd_loss)
        self.acc_metric.update_state(y, tf.nn.softmax(student_logits, axis=1))
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        (x, teacher_probs), y = data
        student_logits = self.student(x, training=False)
        T = tf.cast(self.temperature, tf.float32)
        s_loss = self.student_loss_fn(y, student_logits)
        student_soft = tf.nn.softmax(student_logits / T, axis=1)
        teacher_soft = tf.nn.softmax(teacher_probs / T, axis=1)
        kd_loss = self.kd_loss_fn(teacher_soft, student_soft) * (T * T)
        loss = self.alpha * s_loss + (1.0 - self.alpha) * kd_loss
        self.total_loss_tracker.update_state(loss)
        self.student_loss_tracker.update_state(s_loss)
        self.kd_loss_tracker.update_state(kd_loss)
        self.acc_metric.update_state(y, tf.nn.softmax(student_logits, axis=1))
        return {m.name: m.result() for m in self.metrics}

# ===================== INSTANTIATE DISTILLER =====================
alpha = 0.5
temperature = 3.0
distiller = Distiller(student=student, teacher=teacher, alpha=alpha, temperature=temperature)
opt = Adam(1e-4)
distiller.compile(optimizer=opt)
print("✅ Distiller compiled")

# ===================== CALLBACKS and TRAINING (RUN 1: 25 epochs) =====================
save_dir = "/content/drive/MyDrive/alz_kd_results_run1"
os.makedirs(save_dir, exist_ok=True)
ckpt_path = os.path.join(save_dir, 'best_student_run1.keras')
final_student_path = os.path.join(save_dir, 'final_student_run1.keras')
csv_log = os.path.join(save_dir, 'kd_metrics_run1.csv')

ckpt_cb = tf.keras.callbacks.ModelCheckpoint(ckpt_path, monitor='val_accuracy', mode='max', save_best_only=True, save_weights_only=False, verbose=1)
csv_cb = tf.keras.callbacks.CSVLogger(csv_log)

epochs_run1 = 50

print(f"\n🚀 Starting Run 1 training for {epochs_run1} epochs ...")
history = distiller.fit(
    train_seq,
    validation_data=val_seq,
    epochs=epochs_run1,
    callbacks=[ckpt_cb, csv_cb]
)

# ===================== SAVE FINAL STUDENT (softmax wrapper for inference) =====================
final_student = tf.keras.Sequential([distiller.student, Softmax(name='probabilities')])
final_student.save(final_student_path)
print(f"✅ Saved final student (Run1) to: {final_student_path}")

# Save artifacts message
print("Run 1 complete. Saved:")
print(" - filelist_mapping.csv")
print(" - teacher_probs_all.npy")
print(" - best_student_run1.keras")
print(" - final_student_run1.keras")
print(" - kd_metrics_run1.csv")

✅ Teacher loaded and frozen
🔹 Generating teacher predictions for all files (this may take time)...


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1262/1262 ━━━━━━━━━━━━━━━━━━━━ 187s 136ms/step
✅ Teacher predictions generated
Saved teacher predictions and mapping to: /content/drive/MyDrive/alz_kd_results_run1
Train samples: 32307, Val samples: 8077
✅ Distiller compiled

🚀 Starting Run 1 training for 50 epochs ...
Epoch 1/50


/usr/local/lib/python3.12/dist-packages/keras/src/backend/tensorflow/nn.py:675: UserWarning: "`categorical_crossentropy` received `from_logits=True`, but the `output` argument was produced by a Softmax activation and thus does not represent logits. Was this intended?
  output, from_logits = _get_logits(


1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 87ms/step - accuracy: 0.3523 - kd_loss: 0.0442 - loss: 0.9762 - student_loss: 1.9083
Epoch 1: val_accuracy improved from -inf to 0.56840, saving model to /content/drive/MyDrive/alz_kd_results_run1/best_student_run1.keras


/usr/local/lib/python3.12/dist-packages/keras/src/saving/saving_api.py:107: UserWarning: You are saving a model that has not yet been built. It might not contain any weights yet. Consider building the model first by calling it on some data.
  return saving_lib.save_model(model, filepath)


1010/1010 ━━━━━━━━━━━━━━━━━━━━ 135s 111ms/step - accuracy: 0.3523 - kd_loss: 0.0442 - loss: 0.9761 - student_loss: 1.9081 - val_accuracy: 0.5684 - val_kd_loss: 0.0398 - val_loss: 0.5159 - val_student_loss: 0.9919
Epoch 2/50
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 73ms/step - accuracy: 0.5077 - kd_loss: 0.0449 - loss: 0.6345 - student_loss: 1.2242
Epoch 2: val_accuracy improved from 0.56840 to 0.68503, saving model to /content/drive/MyDrive/alz_kd_results_run1/best_student_run1.keras
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 86s 85ms/step - accuracy: 0.5077 - kd_loss: 0.0449 - loss: 0.6345 - student_loss: 1.2241 - val_accuracy: 0.6850 - val_kd_loss: 0.0442 - val_loss: 0.3812 - val_student_loss: 0.7182
Epoch 3/50
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 0s 71ms/step - accuracy: 0.6061 - kd_loss: 0.0476 - loss: 0.4834 - student_loss: 0.9192
Epoch 3: val_accuracy improved from 0.68503 to 0.72329, saving model to /content/drive/MyDrive/alz_kd_results_run1/best_student_run1.keras
1010/1010 ━━━━━━━━━━━━━━━━━━━━ 85s 84m